In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
base_dir = Path("/scratch/jma819/CaliAli_linearTrackData")

peak_stats_paths = [
    base_dir / "m311_analysis/all_peak_stats_20260518_211705.csv",
    base_dir / "m388_analysis/all_peak_stats_20260518_211705.csv",
    # add more here
]

def summarize_peak_stats_file(
    csv_path,
    amp_col="amp_max",
    length_col="length_samples",
    chunksize=50000,
    sampling_period_s=None,   # e.g. 0.05 if you want seconds too
):
    csv_path = Path(csv_path)
    mouse = csv_path.parent.name.replace("_analysis", "")

    n_events = 0
    amp_sum = 0.0
    length_sum = 0.0

    for chunk in pd.read_csv(
        csv_path,
        usecols=["cell", amp_col, length_col],
        chunksize=chunksize,
        low_memory=False,
    ):
        chunk[amp_col] = pd.to_numeric(chunk[amp_col], errors="coerce")
        chunk[length_col] = pd.to_numeric(chunk[length_col], errors="coerce")

        valid = chunk[[amp_col, length_col]].dropna()
        if valid.empty:
            continue

        n_events += len(valid)
        amp_sum += valid[amp_col].sum()
        length_sum += valid[length_col].sum()

    if n_events == 0:
        return {
            "mouse": mouse,
            "peak_stats_csv": str(csv_path),
            "n_events": 0,
            "mean_amp_max": np.nan,
            "mean_length_samples": np.nan,
            "mean_length_s": np.nan if sampling_period_s is not None else None,
        }

    mean_amp = amp_sum / n_events
    mean_length_samples = length_sum / n_events

    out = {
        "mouse": mouse,
        "peak_stats_csv": str(csv_path),
        "n_events": n_events,
        "mean_amp_max": mean_amp,
        "mean_length_samples": mean_length_samples,
    }

    if sampling_period_s is not None:
        out["mean_length_s"] = mean_length_samples * sampling_period_s

    return out

summary_rows = [
    summarize_peak_stats_file(path, sampling_period_s=0.05)
    for path in peak_stats_paths
]

peak_stats_mouse_summary_df = pd.DataFrame(summary_rows)
peak_stats_mouse_summary_df

,mouse,peak_stats_csv,n_events,mean_amp_max,mean_length_samples,mean_length_s
0,m311,/scratch/jma819/CaliAli_linearTrackData/m311_a...,249059,9.930330,20211.969369,1010.598468
1,m388,/scratch/jma819/CaliAli_linearTrackData/m388_a...,273566,9.664051,13834.559580,691.727979
